In [1]:
# =====================================================================
# OmniVoice TTS — self-contained WebSocket test (edit params & re-run)
# =====================================================================
# %pip install -q websockets   # uncomment if websockets isn't in this kernel

import asyncio, json, time, uuid, wave
import websockets
from IPython.display import Audio, display

In [ ]:

# ---- params ----
HOST, PORT, PATH_PREFIX = "172.16.1.4:80", 8080, "/"   # via nginx
# direct (no nginx): HOST, PORT, PATH_PREFIX = "127.0.0.1", 8080, ""
VOICE   = "hausa_ma"
LANG    = "ha"        # match the text's language: as=Assamese, bn=Bengali, hi=Hindi, en=English
SPEED   = 1.0        # e.g. 1.1
OUT     = "hausa-ma-1.wav"
TEXT    = "Don Allah, faɗi lambobi huɗu na ƙarshe na lambar asusunku domin in tabbatar da bayananku."
# TEXT    = "hello, i'm aarav from big basket. rohil asked brahmanandam reddy to inform avinash that swastik, pranav, and chandrasekhar should come over, so akshay, vignesh, and adithyan can address the issue with kirtiman, raghav, and krishnakumar."

In [6]:
sentence_json = {
  # "pcm": [
  #   "Hello! Welcome to GTBank. My name na Gigi, your virtual assistant. Wetin I fit help you with today?",
  #   "Abeg, tell me the last four digits of your account number so I fit verify your account.",
  #   "You wan transfer money? Abeg tell me the account number, bank name, and the amount wey you wan send.",
  #   "If your ATM card don loss or you wan block am, tell me now and I go help you do am immediately."
  # ],
  # "yo": [
  #   "Ẹ káàbọ̀ sí GTBank. Orúkọ mi ni Gigi, olùrànlọ́wọ́ foju kọ̀ǹpútà yín. Báwo ni mo ṣe lè ràn yín lọ́wọ́ lónìí?",
  #   "Jọ̀wọ́, sọ díjítì mẹ́rin ìkẹyìn ti nọ́mbà àkọọ́ntì yín kí n lè jẹ́rìí ìdánimọ̀ yín.",
  #   "Ṣé ẹ fẹ́ fi owó ránṣẹ́? Jọ̀wọ́ sọ nọ́mbà àkọọ́ntì, orúkọ ilé-ifowopamọ́, àti iye owó tí ẹ fẹ́ rán.",
  #   "Bí ẹ bá pàdánù káàdì ATM yín tàbí ẹ fẹ́ dí i, jọ̀wọ́ sọ fún mi, màá ràn yín lọ́wọ́ lẹ́sẹ̀kẹsẹ̀."
  # ],
  # "ha": [
  #   "Sannu! Barka da zuwa GTBank. Sunana Gigi, mataimakiyarku ta kwamfuta. Ta yaya zan iya taimaka muku yau?",
  #   "Don Allah, faɗi lambobi huɗu na ƙarshe na lambar asusunku domin in tabbatar da bayananku.",
  #   "Kuna son tura kuɗi? Don Allah ku faɗi lambar asusu, sunan banki, da adadin kuɗin da kuke son turawa.",
  #   "Idan katin ATM ɗinku ya ɓace ko kuna son a dakatar da shi, ku sanar da ni yanzu, zan taimaka muku nan take."
  # ],
  # "ig": [
  #   "Ndeewo! Nnọọ na GTBank. Aha m bụ Gigi, onye enyemaka gị na-arụ ọrụ n'ịntanetị. Kedu ka m ga-esi nyere gị taa?",
  #   "Biko, gwa m ọnụọgụ anọ ikpeazụ nke nọmba akaụntụ gị ka m nwee ike kwado njirimara gị.",
  #   "Ị chọrọ izipu ego? Biko, gwa m nọmba akaụntụ, aha ụlọ akụ, na ego ịchọrọ izipu.",
  #   "Ọ bụrụ na kaadị ATM gị efuola ma ọ bụ ịchọrọ igbochi ya, gwa m ugbu a, aga m enyere gị aka ozugbo."
  # ],
  # "en": [
  #   "Hello! Welcome to GTBank. My name is Gigi, your virtual assistant. How may I assist you today?",
  #   "Please provide the last four digits of your account number so I can verify your account.",
  #   "Would you like to make a transfer? Please provide the recipient's account number, bank name, and the amount you would like to send.",
  #   "If your ATM card has been lost, stolen, or you would like to block it, please let me know and I'll assist you right away."
  # ],
  "pcm": [
      "Hello! Welcome to GTBank. My name na Gigi, your virtual assistant. Wetin I fit help you with today?",
      "Abeg, tell me the last four digits of your account number make I fit verify your account.",
      "You wan transfer money? Abeg tell me the account number, bank name, and the amount wey you wan send.",
      "If your ATM card don loss, don spoil, or you wan block am, just tell me and I go help you do am immediately."
    ]
}

In [9]:
voice_json = {
  "ha": {"female":["hausa-female-1"],"male":["hausa-male-1","hausa-male-2"]},
  "ig": {"female":["igbo-female-1","igbo-female-2","igbo-female-3"],"male":["igbo-male-1","igbo-male-2"]},
  "en": {"female":["igbo-female-1","hausa-female-1","yoruba-female-1,yoruba-female-2","igbo-female-2","igbo-female-3"]},
  "pcm": {"female":["igbo-female-1","hausa-female-1","yoruba-female-1,yoruba-female-2","igbo-female-2","igbo-female-3"]},
  "yo":{"female":["yoruba-female-1,yoruba-female-2"]}
}

In [16]:

def _split(raw):
    """Split a combined binary frame ({json header} + raw PCM) -> (msg, pcm_bytes)."""
    if isinstance(raw, str):
        return json.loads(raw), b""
    depth = end = 0
    for i, b in enumerate(raw):
        if b == 0x7B: depth += 1          # '{'
        elif b == 0x7D:                   # '}'
            depth -= 1
            if depth == 0:
                end = i + 1; break
    return json.loads(raw[:end]), raw[end:]

async def synth(text, voice=None, lang=None, speed=None, out_file=None):
    call_id = f"nb-{uuid.uuid4().hex[:8]}"
    url = f"ws://{HOST}:{PORT}{PATH_PREFIX.rstrip('/')}/ws/{call_id}"
    print("connecting", url)
    pcm, sr, ttfb, saw_final, done = bytearray(), 24000, None, False, {}
    req = {"type": "synthesize", "call_id": call_id, "text_id": uuid.uuid4().hex[:8],
           "text": text, "streaming": True}
    if voice: req["voice_id"] = voice
    if lang:  req["language"] = lang
    if speed: req["speed"] = speed

    async with websockets.connect(url, max_size=100*1024*1024, open_timeout=10,
                                  close_timeout=3, ping_interval=None) as ws:
        t0 = time.perf_counter()
        await ws.send(json.dumps(req))
        while True:
            try:
                raw = await asyncio.wait_for(ws.recv(), timeout=(5 if saw_final else 120))
            except asyncio.TimeoutError:
                if saw_final: break
                print("TIMEOUT waiting for audio"); return None
            msg, audio = _split(raw)
            mt = msg.get("type")
            if mt == "audio_chunk":
                if not audio:
                    audio = await ws.recv()
                    if isinstance(audio, str): audio = audio.encode()
                if ttfb is None:
                    ttfb = time.perf_counter() - t0
                    print(f"first chunk in {ttfb*1000:.0f} ms  cache_hit={msg.get('cache_hit')}")
                sr = msg.get("sample_rate", sr); pcm += audio
                if msg.get("is_final"): saw_final = True
            elif mt == "audio_done":
                done = msg; break
            elif mt == "error":
                print("SERVER ERROR:", msg.get("error")); return None

    total = time.perf_counter() - t0
    audio_s = (len(pcm)//2)/sr if sr else 0
    print(f"done  chunks={done.get('chunks')}  audio={audio_s:.2f}s  "
          f"ttfb={(ttfb*1000 if ttfb else 0):.0f}ms  total={total*1000:.0f}ms  "
          f"rtf={done.get('rtf')}  bytes={len(pcm)}  sr={sr}")
    with wave.open(out_file or OUT, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes(bytes(pcm))
    print("wrote", OUT)
    return OUT

In [17]:

out = await synth(text=TEXT, voice=VOICE, lang=LANG, speed=SPEED)          # Jupyter supports top-level await
if out: display(Audio(filename=out))

connecting ws://101.53.141.123:8080/ws/nb-f6ed926b
first chunk in 684 ms  cache_hit=False
done  chunks=1  audio=6.54s  ttfb=684ms  total=733ms  rtf=None  bytes=313964  sr=24000
wrote hausa-ma-1.wav


In [6]:
# =====================================================================
# Batch synth: run TTS for every language in sentence_json with the
# matching voices from voice_json, saving in the Yoruba/ sample layout:
#     <out_root>/<LangName>/<gender>/<gender><n>/audio<m>.wav
# =====================================================================
import os

LANG_NAMES = {
    "yo": "Yoruba", "ha": "Hausa", "ig": "Igbo", "pcm": "Pidgin",
    "en": "English", "hi": "Hindi", "bn": "Bengali", "as": "Assamese",
}

def _voice_ids(entry):
    """Flatten a gender's voice list, tolerating comma-joined strings
    like ['a,b'] which really mean two voices."""
    ids = []
    for v in entry:
        ids.extend(p.strip() for p in str(v).split(",") if p.strip())
    return ids

async def synth_all(sentence_json, voice_json, out_root=".", speed=SPEED):
    """For each language in sentence_json, synthesize every sentence with
    each configured voice (per gender) and write to the Yoruba/ layout.
    Returns the list of files written."""
    written = []
    for lang, sentences in sentence_json.items():
        lang_name = LANG_NAMES.get(lang, lang)
        genders = voice_json.get(lang)
        if not genders:
            print(f"[skip] no voices configured for lang={lang!r}")
            continue
        for gender, entry in genders.items():
            for vi, voice_id in enumerate(_voice_ids(entry), start=1):
                vdir = os.path.join(out_root, lang_name, gender, f"{gender}{vi}")
                os.makedirs(vdir, exist_ok=True)
                for si, text in enumerate(sentences, start=1):
                    out_file = os.path.join(vdir, f"audio{si}.wav")
                    print(f"\n== {lang_name} | {gender}{vi} ({voice_id}) | "
                          f"sentence {si}/{len(sentences)} -> {out_file}")
                    ok = await synth(text, voice=voice_id, lang=lang,
                                     speed=speed, out_file=out_file)
                    if ok:
                        written.append(out_file)
    print(f"\nDONE — wrote {len(written)} files")
    return written

In [40]:
# Run the batch. out_root="." writes next to the existing Yoruba/ sample
# (assuming this notebook runs from the sample_files/ directory).
files = await synth_all(sentence_json, voice_json, out_root=".")


== English | female1 (igbo-female-1) | sentence 1/4 -> ./English/female/female1/audio1.wav
connecting ws://172.16.1.4:80/omnivoice-tts/ws/nb-33bfdf70
first chunk in 588 ms  cache_hit=False
done  chunks=2  audio=7.42s  ttfb=588ms  total=614ms  rtf=0.074  bytes=356160  sr=24000
wrote yoruba-male-2-sample4.wav

== English | female1 (igbo-female-1) | sentence 2/4 -> ./English/female/female1/audio2.wav
connecting ws://172.16.1.4:80/omnivoice-tts/ws/nb-4239733c
first chunk in 485 ms  cache_hit=False
done  chunks=1  audio=6.87s  ttfb=485ms  total=494ms  rtf=0.064  bytes=329760  sr=24000
wrote yoruba-male-2-sample4.wav

== English | female1 (igbo-female-1) | sentence 3/4 -> ./English/female/female1/audio3.wav
connecting ws://172.16.1.4:80/omnivoice-tts/ws/nb-10092abe
first chunk in 563 ms  cache_hit=False
done  chunks=2  audio=9.97s  ttfb=563ms  total=583ms  rtf=0.047  bytes=478560  sr=24000
wrote yoruba-male-2-sample4.wav

== English | female1 (igbo-female-1) | sentence 4/4 -> ./English/fema

In [ ]:
# =====================================================================
# Regenerate ONLY the flagged samples (semantic WER too high) into a
# SEPARATE folder, preserving the same <LangName>/<gender>/<gender><n>/
# layout so they drop back in / compare 1:1 with the originals.
#
# Self-contained: the original ha/ig/yo sentences are embedded here so
# this works regardless of what the params cells above currently hold.
# voice_id is derived from the path as "<langname>-<gender>-<n>",
# matching the convention in voice_json (e.g. igbo-female-2).
# =====================================================================
import os, re

# Full folder name -> language code
_CODE_BY_LANG_NAME = {"Yoruba": "yo", "Hausa": "ha", "Igbo": "ig"}

# The exact sentences the flagged samples were generated from.
REDO_SENTENCES = {
    "yo": [
        "Ẹ káàbọ̀ sí GTBank. Orúkọ mi ni Gigi, olùrànlọ́wọ́ foju kọ̀ǹpútà yín. Báwo ni mo ṣe lè ràn yín lọ́wọ́ lónìí?",
        "Jọ̀wọ́, sọ díjítì mẹ́rin ìkẹyìn ti nọ́mbà àkọọ́ntì yín kí n lè jẹ́rìí ìdánimọ̀ yín.",
        "Ṣé ẹ fẹ́ fi owó ránṣẹ́? Jọ̀wọ́ sọ nọ́mbà àkọọ́ntì, orúkọ ilé-ifowopamọ́, àti iye owó tí ẹ fẹ́ rán.",
        "Bí ẹ bá pàdánù káàdì ATM yín tàbí ẹ fẹ́ dí i, jọ̀wọ́ sọ fún mi, màá ràn yín lọ́wọ́ lẹ́sẹ̀kẹsẹ̀.",
    ],
    "ha": [
        "Sannu! Barka da zuwa GTBank. Sunana Gigi, mataimakiyarku ta kwamfuta. Ta yaya zan iya taimaka muku yau?",
        "Don Allah, faɗi lambobi huɗu na ƙarshe na lambar asusunku domin in tabbatar da bayananku.",
        "Kuna son tura kuɗi? Don Allah ku faɗi lambar asusu, sunan banki, da adadin kuɗin da kuke son turawa.",
        "Idan katin ATM ɗinku ya ɓace ko kuna son a dakatar da shi, ku sanar da ni yanzu, zan taimaka muku nan take.",
    ],
    "ig": [
        "Ndeewo! Nnọọ na GTBank. Aha m bụ Gigi, onye enyemaka gị na-arụ ọrụ n'ịntanetị. Kedu ka m ga-esi nyere gị taa?",
        "Biko, gwa m ọnụọgụ anọ ikpeazụ nke nọmba akaụntụ gị ka m nwee ike kwado njirimara gị.",
        "Ị chọrọ izipu ego? Biko, gwa m nọmba akaụntụ, aha ụlọ akụ, na ego ịchọrọ izipu.",
        "Ọ bụrụ na kaadị ATM gị efuola ma ọ bụ ịchọrọ igbochi ya, gwa m ugbu a, aga m enyere gị aka ozugbo.",
    ],
    "pcm": [
    "Hello! Welcome to GTBank. My name na Gigi, your virtual assistant. Wetin I fit help you with today?",
    "Abeg, tell me the last four digits of your account number make I fit verify your account.",
    "You wan transfer money? Abeg tell me the account number, bank name, and the amount wey you wan send.",
    "If your ATM card don loss, don spoil, or you wan block am, just tell me and I go help you do am immediately."
  ]
}

# Original audio paths flagged for regeneration (from the WER report).
REDO = [
    "Hausa/male/male2/audio1.wav",
    "Hausa/male/male1/audio1.wav",
    "Hausa/male/male2/audio2.wav",
    "Hausa/male/male2/audio3.wav",
    "Hausa/male/male2/audio4.wav",
    "Hausa/male/male1/audio2.wav",
    "Igbo/female/female3/audio3.wav",
    "Igbo/male/male2/audio1.wav",
    "Igbo/male/male2/audio3.wav",
    "Igbo/female/female2/audio1.wav",
    "Igbo/female/female3/audio1.wav",
    "Igbo/male/male1/audio1.wav",
    "Yoruba/female/female1/audio4.wav",
    "Yoruba/female/female2/audio2.wav",
    "Yoruba/male/male1/audio1.wav",
    "Yoruba/female/female2/audio1.wav",
    "Yoruba/male/male1/audio4.wav",
    "Yoruba/female/female1/audio2.wav",
]

async def redo_samples(redo=REDO, sentences=REDO_SENTENCES,
                       out_root="regenerated", speed=SPEED):
    """Re-synthesize each flagged path into out_root, keeping the same
    language/gender/speaker/sentence-number layout. Returns files written."""
    written, skipped = [], []
    for rel in redo:
        parts = rel.replace("\\", "/").split("/")
        lang_name, gender, speaker, fname = parts[-4], parts[-3], parts[-2], parts[-1]
        lang = _CODE_BY_LANG_NAME.get(lang_name)
        num  = int(re.search(r"(\d+)", speaker).group(1))     # male2 -> 2
        sent = int(re.search(r"(\d+)", fname).group(1))       # audio3.wav -> 3
        voice_id = f"{lang_name.lower()}-{gender}-{num}"       # -> yoruba-male-1
        try:
            text = sentences[lang][sent - 1]
        except (KeyError, IndexError, TypeError):
            print(f"[skip] no sentence for {rel} (lang={lang!r}, idx={sent})")
            skipped.append(rel); continue
        out_file = os.path.join(out_root, lang_name, gender, speaker, f"audio{sent}.wav")
        os.makedirs(os.path.dirname(out_file), exist_ok=True)
        print(f"\n== {lang_name} | {speaker} ({voice_id}) | sentence {sent} -> {out_file}")
        ok = await synth(text, voice=voice_id, lang=lang, speed=speed, out_file=out_file)
        if ok:
            written.append(out_file)
    print(f"\nDONE — regenerated {len(written)}/{len(redo)} files into {out_root!r}")
    if skipped:
        print("Skipped (no matching sentence):", skipped)
    return written

In [11]:
# Regenerate the flagged samples into ./regenerated (separate from the
# originals). Change out_root if the kernel isn't running from sample_files/.
regen = await redo_samples(out_root="regenerated")


== Hausa | male2 (hausa-male-2) | sentence 1 -> regenerated/Hausa/male/male2/audio1.wav
connecting ws://172.16.1.4:80/omnivoice-tts/ws/nb-63b5fa1f
first chunk in 1913 ms  cache_hit=False
done  chunks=2  audio=14.31s  ttfb=1913ms  total=1967ms  rtf=0.128  bytes=686880  sr=24000
wrote yoruba-male-2-sample4.wav

== Hausa | male1 (hausa-male-1) | sentence 1 -> regenerated/Hausa/male/male1/audio1.wav
connecting ws://172.16.1.4:80/omnivoice-tts/ws/nb-d71dd900
first chunk in 1307 ms  cache_hit=False
done  chunks=2  audio=10.28s  ttfb=1307ms  total=1335ms  rtf=0.125  bytes=493440  sr=24000
wrote yoruba-male-2-sample4.wav

== Hausa | male2 (hausa-male-2) | sentence 2 -> regenerated/Hausa/male/male2/audio2.wav
connecting ws://172.16.1.4:80/omnivoice-tts/ws/nb-cb0465e8
first chunk in 1033 ms  cache_hit=False
done  chunks=1  audio=12.39s  ttfb=1033ms  total=1052ms  rtf=0.073  bytes=594720  sr=24000
wrote yoruba-male-2-sample4.wav

== Hausa | male2 (hausa-male-2) | sentence 3 -> regenerated/Hausa/